<a href="https://colab.research.google.com/github/Valt1719/TTNT_VoThanhLong/blob/main/A.I/03.ThuatToanAsao/BTVN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Cài đặt bài toán người giao hàng theo thuật giải A*.
import heapq

def calculate_mst_weight(remaining_cities, dist_matrix):
    """Tính tổng trọng số của Cây khung nhỏ nhất (MST) cho các thành phố còn lại."""
    if not remaining_cities:
        return 0

    nodes = list(remaining_cities)
    visited = {nodes[0]}
    unvisited = set(nodes[1:])
    mst_weight = 0
    edges = []

    # Thuật toán Prim đơn giản để tìm MST
    for to_node in unvisited:
        heapq.heappush(edges, (dist_matrix[nodes[0]][to_node], nodes[0], to_node))

    while unvisited and edges:
        weight, u, v = heapq.heappop(edges)
        if v in unvisited:
            unvisited.remove(v)
            visited.add(v)
            mst_weight += weight
            for next_node in unvisited:
                heapq.heappush(edges, (dist_matrix[v][next_node], v, next_node))

    return mst_weight

def a_star_tsp(dist_matrix):
    n = len(dist_matrix)
    start_node = 0
    # State: (f_score, current_node, visited_tuple)
    # visited_tuple lưu trữ các thành phố đã đi qua theo thứ tự
    initial_visited = (start_node,)

    # Hàng đợi ưu tiên: (f, g, current_node, path)
    pq = [(0, 0, start_node, initial_visited)]

    while pq:
        f, g, u, path = heapq.heappop(pq)

        # Nếu đã đi qua tất cả các thành phố
        if len(path) == n:
            # Cộng thêm đường về điểm xuất phát
            total_dist = g + dist_matrix[u][start_node]
            full_path = path + (start_node,)
            return total_dist, full_path

        # Tìm các thành phố chưa ghé thăm
        remaining = set(range(n)) - set(path)

        for v in remaining:
            new_g = g + dist_matrix[u][v]

            # Tính Heuristic h(n):
            # 1. MST của các thành phố còn lại
            # 2. Khoảng cách từ u đến thành phố gần nhất trong tập remaining
            # 3. Khoảng cách từ thành phố gần nhất trong tập remaining về lại điểm xuất phát
            remaining_after_v = remaining - {v}
            h_mst = calculate_mst_weight(remaining_after_v, dist_matrix)

            # Ước lượng tối ưu hơn (Admissible Heuristic)
            h = h_mst
            if remaining_after_v:
                dist_to_mst = min(dist_matrix[v][r] for r in remaining_after_v)
                dist_from_mst = min(dist_matrix[r][start_node] for r in remaining_after_v)
                h += (dist_to_mst + dist_from_mst)
            else:
                h += dist_matrix[v][start_node]

            new_f = new_g + h
            heapq.heappush(pq, (new_f, new_g, v, path + (v,)))

    return None

# --- Chạy thử nghiệm ---
# Ma trận khoảng cách giữa 4 thành phố
matrix = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

distance, route = a_star_tsp(matrix)
print(f"Tổng quãng đường ngắn nhất: {distance}")
print(f"Lộ trình: {' -> '.join(map(str, route))}")